# statsmodels — Stationarity, cointegration, and lead-lag analysis

## Project goal

On the 4-asset crypto panel, run a complete classical time-series diagnostic: stationarity (ADF + KPSS) of prices and returns, pairwise cointegration matrix, Granger causality between pairs at multiple lags, OLS regression of one return on the others, and a SARIMA fit with full residual diagnostics.


## Why this exercises the cheatsheet

Hits every test in the statsmodels cheatsheet: stationarity duo, Granger causality, Engle-Granger cointegration, formula-style OLS, SARIMAX with Ljung-Box and ACF residual diagnostics, plus heteroscedasticity testing on the OLS residuals.


## Sub-task 1: ADF + KPSS verdict matrix

For each of (BTC, ETH, SOL, BNB), run ADF and KPSS on both the close price and the log return. Combine into a 4-asset × 4-test verdict matrix where each cell is one of `stationary` / `non-stationary` / `trend-stationary` / `indeterminate` based on the (ADF reject?, KPSS reject?) combination.

**Patterns this forces:**

- `adfuller(series)`, `kpss(series, regression='c', nlags='auto')`
- decision matrix from the (reject, reject) combo
- `warnings.filterwarnings('ignore')` for KPSS' interpolation noise


In [ ]:
# Your answer here

## Sub-task 2: Pairwise cointegration matrix

Compute the Engle-Granger cointegration p-value for every pair of (symbol, symbol) on the close prices. Output a 4×4 symmetric DataFrame with p-values. Identify any pair with p < 0.05.

**Patterns this forces:**

- `from statsmodels.tsa.stattools import coint`
- nested loop over symbol pairs (`for s1, s2 in combinations(...)`)
- DataFrame indexed by symbol, columns by symbol


In [ ]:
# Your answer here

## Sub-task 3: Granger causality at multiple lags

Test whether ETH's hourly return Granger-causes BTC's hourly return at lags 1, 4, and 24. Report the p-value at each lag. Then test the reverse direction. Comment on which direction (if any) shows predictive power.

**Patterns this forces:**

- `grangercausalitytests(df[[y, x]], maxlag=24)`
- remember column order: `[y, x]` tests x → y, NOT y → x
- `results[lag][0]['ssr_ftest'][1]` for the F-test p-value


In [ ]:
# Your answer here

## Sub-task 4: OLS via formula API

Regress BTC hourly log return on the simultaneous returns of ETH, SOL, BNB using the formula API: `smf.ols('btc_ret ~ eth_ret + sol_ret + bnb_ret', data=df).fit()`. Print the `.summary().tables[1]` to see coefficients, p-values, t-stats.

**Patterns this forces:**

- `import statsmodels.formula.api as smf`
- wide-format frame with one column per symbol's return
- `.summary().tables[1]` for the coefficient block


In [ ]:
# Your answer here

## Sub-task 5: Heteroscedasticity test on OLS residuals

Take the OLS residuals from sub-task 4. Run a Breusch-Pagan test (`het_breuschpagan`). Interpret: is the variance of BTC's residual constant across the regressors, or does it grow with them?

**Patterns this forces:**

- `from statsmodels.stats.diagnostic import het_breuschpagan`
- interpretation: p < 0.05 ⇒ heteroscedastic ⇒ use HC robust SEs next time
- comment: would refit with `cov_type='HC3'` if rejected


In [ ]:
# Your answer here

## Sub-task 6: SARIMA fit + residual diagnostics

Fit a SARIMAX(1, 0, 1)(1, 0, 1, 24) model to BTC log returns. After fitting: (a) Ljung-Box on residuals at lags [1, 12, 24], (b) plot the residual ACF up to 48 lags, (c) interpret: do residuals look white? If not, what would you do?

**Patterns this forces:**

- `SARIMAX(y, order=(1,0,1), seasonal_order=(1,0,1,24)).fit(disp=False)`
- `acorr_ljungbox(resid, lags=[1, 12, 24], return_df=True)`
- `plot_acf(resid, lags=48)`


In [ ]:
# Your answer here

## What success looks like

- Verdict matrix from sub-task 1 — every cell explicit (no NaNs).
- 4×4 cointegration matrix, with at least one pair flagged p < 0.05 (or you state that none are).
- A SARIMA fit with Ljung-Box p > 0.05 at every tested lag — i.e. residuals look white. If not, you've documented why and what would change.
- One paragraph at the end synthesising: *'on this universe, what's the evidence for predictable lead-lag structure, and what's the strongest model that captures it?'*
